# TC4 钛合金阿伦纽斯（Arrhenius）本构方程计算

基于 Sellars-Tegart 双曲正弦模型：
$$\dot{\varepsilon} = A [\sinh(\alpha \sigma)]^n \exp\left(-\frac{Q}{RT}\right)$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

R_GAS = 8.314  # J/(mol·K)

## 1. 数据加载与清洗

从原始 Excel 文件正确读取数据（每个 sheet 为一个温度，每两列为一组应变-应力数据对）

In [ ]:
file_path = '/Users/bertonyang/project/chenglu/data/TC4_0219.xlsx'
xlsx = pd.ExcelFile(file_path)

all_data = []
for sheet_name in xlsx.sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    temp = int(sheet_name)
    num_pairs = df.shape[1] // 2
    for i in range(num_pairs):
        strain_rate = float(df.iloc[0, i * 2])
        col_strain = i * 2
        col_stress = i * 2 + 1
        strain = pd.to_numeric(df.iloc[1:, col_strain], errors='coerce')
        stress = pd.to_numeric(df.iloc[1:, col_stress], errors='coerce')
        for s_val, r_val in zip(strain, stress):
            if pd.notna(s_val) and pd.notna(r_val):
                all_data.append({
                    '温度_C': temp,
                    '温度_K': temp + 273.15,
                    '应变速率': strain_rate,
                    '真应变': s_val,
                    '流变应力': r_val
                })

df_all = pd.DataFrame(all_data)

# 过滤掉应力 <= 0 的数据点（物理上无意义）
df_all = df_all[df_all['流变应力'] > 0].copy()
# 过滤掉应变 <= 0 的数据点
df_all = df_all[df_all['真应变'] > 0].copy()

print(f'总数据量: {len(df_all)} 条')
print(f'温度(°C): {sorted(df_all["温度_C"].unique())}')
print(f'应变速率(s⁻¹): {sorted(df_all["应变速率"].unique())}')
print()
print('各条件数据量:')
for temp in sorted(df_all['温度_C'].unique()):
    for sr in sorted(df_all['应变速率'].unique()):
        sub = df_all[(df_all['温度_C'] == temp) & (df_all['应变速率'] == sr)]
        if len(sub) > 0:
            print(f'  T={temp}°C, ε̇={sr} s⁻¹: {len(sub)} 行, '
                  f'应变=[{sub["真应变"].min():.4f}, {sub["真应变"].max():.4f}], '
                  f'应力=[{sub["流变应力"].min():.2f}, {sub["流变应力"].max():.2f}] MPa')

## 2. 绘制真应力-真应变曲线

In [ ]:
temps = sorted(df_all['温度_C'].unique())
rates = sorted(df_all['应变速率'].unique())
colors = plt.cm.coolwarm(np.linspace(0, 1, len(temps)))

fig, axes = plt.subplots(1, len(rates), figsize=(5 * len(rates), 4), sharey=False)
if len(rates) == 1:
    axes = [axes]
for ax, sr in zip(axes, rates):
    for temp, c in zip(temps, colors):
        sub = df_all[(df_all['温度_C'] == temp) & (df_all['应变速率'] == sr)].sort_values('真应变')
        if len(sub) > 0:
            ax.plot(sub['真应变'], sub['流变应力'], color=c, label=f'{temp}°C', linewidth=0.8)
    ax.set_title(f'ε̇ = {sr} s⁻¹')
    ax.set_xlabel('真应变')
    ax.set_ylabel('流变应力 (MPa)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 离散应变点处的应力插值

在每个（温度，应变速率）条件下，对应力-应变曲线进行插值，提取离散应变点处的流变应力值。

In [ ]:
strain_points = np.arange(0.05, 0.65, 0.05)
print(f'离散应变点: {strain_points}')
print(f'共 {len(strain_points)} 个应变点')
print()

stress_table = {}
missing_conditions = []

for temp in temps:
    for sr in rates:
        sub = df_all[(df_all['温度_C'] == temp) & (df_all['应变速率'] == sr)].sort_values('真应变')
        if len(sub) < 5:
            missing_conditions.append((temp, sr))
            continue
        # 对数据去重（同一应变值取平均应力）
        sub_grouped = sub.groupby('真应变')['流变应力'].mean().reset_index()
        sub_grouped = sub_grouped.sort_values('真应变')
        try:
            f_interp = interp1d(sub_grouped['真应变'], sub_grouped['流变应力'],
                               kind='linear', fill_value='extrapolate')
            for eps in strain_points:
                if sub_grouped['真应变'].min() <= eps <= sub_grouped['真应变'].max():
                    sigma = float(f_interp(eps))
                    if sigma > 0:
                        stress_table[(temp, sr, eps)] = sigma
        except Exception as e:
            print(f'  插值失败: T={temp}, SR={sr}: {e}')

if missing_conditions:
    print(f'数据不足的条件: {missing_conditions}')

# 构建DataFrame展示
rows = []
for (temp, sr, eps), sigma in stress_table.items():
    rows.append({'温度_C': temp, '温度_K': temp + 273.15, '应变速率': sr, '真应变': eps, '流变应力': sigma})
df_discrete = pd.DataFrame(rows)

print(f'\n插值结果总数: {len(df_discrete)} 条')
print('\n示例数据（ε=0.3 处各条件的流变应力）：')
pivot_sample = df_discrete[df_discrete['真应变'] == 0.3].pivot_table(
    index='温度_C', columns='应变速率', values='流变应力')
print(pivot_sample.round(2))

## 4. 第一步：求解应力乘子 α

### 4.1 低应力近似 → 求 n₁

$$n_1 = \left( \frac{\partial \ln\dot{\varepsilon}}{\partial \ln\sigma} \right)_T$$

以 ln(σ) 为横坐标，ln(ε̇) 为纵坐标，各温度下拟合直线的平均斜率即为 n₁

In [ ]:
n1_results = {}  # {eps: n1_value}
n1_all_slopes = {}  # {eps: {temp: slope}}

for eps in strain_points:
    slopes = {}
    for temp in temps:
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['温度_C'] == temp)]
        if len(sub) >= 2:
            x = np.log(sub['流变应力'].values)
            y = np.log(sub['应变速率'].values)
            slope, intercept, r, p, se = linregress(x, y)
            slopes[temp] = slope
    if slopes:
        n1_results[eps] = np.mean(list(slopes.values()))
        n1_all_slopes[eps] = slopes

print('=== n₁ 计算结果 ===')
print(f'{"应变":>6s} | {"n₁":>10s} | 各温度斜率')
print('-' * 70)
for eps in strain_points:
    if eps in n1_results:
        slope_str = ', '.join([f'{t}°C:{s:.4f}' for t, s in n1_all_slopes[eps].items()])
        print(f'{eps:>6.2f} | {n1_results[eps]:>10.4f} | {slope_str}')

In [ ]:
# 可视化 n₁ 拟合（选取几个典型应变点）
sample_strains = [0.1, 0.3, 0.5]
fig, axes = plt.subplots(1, len(sample_strains), figsize=(5 * len(sample_strains), 4))

for ax, eps in zip(axes, sample_strains):
    for temp, c in zip(temps, colors):
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['温度_C'] == temp)]
        if len(sub) >= 2:
            x = np.log(sub['流变应力'].values)
            y = np.log(sub['应变速率'].values)
            ax.scatter(x, y, color=c, label=f'{temp}°C', zorder=5)
            slope, intercept, _, _, _ = linregress(x, y)
            x_fit = np.linspace(x.min(), x.max(), 50)
            ax.plot(x_fit, slope * x_fit + intercept, '--', color=c, alpha=0.7)
    ax.set_xlabel('ln(σ)')
    ax.set_ylabel('ln(ε̇)')
    ax.set_title(f'n₁ 拟合 (ε={eps})')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 高应力近似 → 求 β

$$\beta = \left( \frac{\partial \ln\dot{\varepsilon}}{\partial \sigma} \right)_T$$

以 σ 为横坐标，ln(ε̇) 为纵坐标，各温度下拟合直线的平均斜率即为 β

In [ ]:
beta_results = {}
beta_all_slopes = {}

for eps in strain_points:
    slopes = {}
    for temp in temps:
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['温度_C'] == temp)]
        if len(sub) >= 2:
            x = sub['流变应力'].values
            y = np.log(sub['应变速率'].values)
            slope, intercept, r, p, se = linregress(x, y)
            slopes[temp] = slope
    if slopes:
        beta_results[eps] = np.mean(list(slopes.values()))
        beta_all_slopes[eps] = slopes

print('=== β 计算结果 ===')
print(f'{"应变":>6s} | {"β":>10s} | 各温度斜率')
print('-' * 70)
for eps in strain_points:
    if eps in beta_results:
        slope_str = ', '.join([f'{t}°C:{s:.6f}' for t, s in beta_all_slopes[eps].items()])
        print(f'{eps:>6.2f} | {beta_results[eps]:>10.6f} | {slope_str}')

In [ ]:
# 可视化 β 拟合
fig, axes = plt.subplots(1, len(sample_strains), figsize=(5 * len(sample_strains), 4))

for ax, eps in zip(axes, sample_strains):
    for temp, c in zip(temps, colors):
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['温度_C'] == temp)]
        if len(sub) >= 2:
            x = sub['流变应力'].values
            y = np.log(sub['应变速率'].values)
            ax.scatter(x, y, color=c, label=f'{temp}°C', zorder=5)
            slope, intercept, _, _, _ = linregress(x, y)
            x_fit = np.linspace(x.min(), x.max(), 50)
            ax.plot(x_fit, slope * x_fit + intercept, '--', color=c, alpha=0.7)
    ax.set_xlabel('σ (MPa)')
    ax.set_ylabel('ln(ε̇)')
    ax.set_title(f'β 拟合 (ε={eps})')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 计算 α = β / n₁

In [ ]:
alpha_results = {}
print('=== α 计算结果 ===')
print(f'{"应变":>6s} | {"n₁":>10s} | {"β":>12s} | {"α":>12s}')
print('-' * 50)
for eps in strain_points:
    if eps in n1_results and eps in beta_results:
        alpha = beta_results[eps] / n1_results[eps]
        alpha_results[eps] = alpha
        print(f'{eps:>6.2f} | {n1_results[eps]:>10.4f} | {beta_results[eps]:>12.6f} | {alpha:>12.6f}')

print(f'\nα 平均值: {np.mean(list(alpha_results.values())):.6f}')

## 5. 第二步：求解主应力指数 n

$$n = \left( \frac{\partial \ln\dot{\varepsilon}}{\partial \ln[\sinh(\alpha \sigma)]} \right)_T$$

以 ln[sinh(ασ)] 为横坐标，ln(ε̇) 为纵坐标，不同温度下拟合直线的平均斜率

In [ ]:
n_results = {}
n_all_slopes = {}

for eps in strain_points:
    if eps not in alpha_results:
        continue
    alpha = alpha_results[eps]
    slopes = {}
    for temp in temps:
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['温度_C'] == temp)]
        if len(sub) >= 2:
            sigma = sub['流变应力'].values
            x = np.log(np.sinh(alpha * sigma))
            y = np.log(sub['应变速率'].values)
            if np.all(np.isfinite(x)) and np.all(np.isfinite(y)):
                slope, intercept, r, p, se = linregress(x, y)
                slopes[temp] = slope
    if slopes:
        n_results[eps] = np.mean(list(slopes.values()))
        n_all_slopes[eps] = slopes

print('=== n（主应力指数）计算结果 ===')
print(f'{"应变":>6s} | {"n":>10s} | 各温度斜率')
print('-' * 80)
for eps in strain_points:
    if eps in n_results:
        slope_str = ', '.join([f'{t}°C:{s:.4f}' for t, s in n_all_slopes[eps].items()])
        print(f'{eps:>6.2f} | {n_results[eps]:>10.4f} | {slope_str}')

In [ ]:
# 可视化 n 拟合
fig, axes = plt.subplots(1, len(sample_strains), figsize=(5 * len(sample_strains), 4))

for ax, eps in zip(axes, sample_strains):
    alpha = alpha_results.get(eps)
    if alpha is None:
        continue
    for temp, c in zip(temps, colors):
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['温度_C'] == temp)]
        if len(sub) >= 2:
            sigma = sub['流变应力'].values
            x = np.log(np.sinh(alpha * sigma))
            y = np.log(sub['应变速率'].values)
            if np.all(np.isfinite(x)):
                ax.scatter(x, y, color=c, label=f'{temp}°C', zorder=5)
                slope, intercept, _, _, _ = linregress(x, y)
                x_fit = np.linspace(x.min(), x.max(), 50)
                ax.plot(x_fit, slope * x_fit + intercept, '--', color=c, alpha=0.7)
    ax.set_xlabel('ln[sinh(ασ)]')
    ax.set_ylabel('ln(ε̇)')
    ax.set_title(f'n 拟合 (ε={eps})')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 第三步：求解热变形激活能 Q

$$Q = R \cdot n \cdot \left( \frac{\partial \ln[\sinh(\alpha \sigma)]}{\partial (1/T)} \right)_{\dot{\varepsilon}}$$

以 1/T 为横坐标，ln[sinh(ασ)] 为纵坐标，在相同应变速率下拟合直线的斜率平均值为 S，则 Q = R × n × S

In [ ]:
Q_results = {}
S_results = {}  # 斜率 S
S_all_slopes = {}

for eps in strain_points:
    if eps not in alpha_results or eps not in n_results:
        continue
    alpha = alpha_results[eps]
    n_val = n_results[eps]
    slopes = {}
    for sr in rates:
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['应变速率'] == sr)]
        if len(sub) >= 2:
            sigma = sub['流变应力'].values
            T_K = sub['温度_K'].values
            x = 1.0 / T_K  # 1/T
            y = np.log(np.sinh(alpha * sigma))
            if np.all(np.isfinite(x)) and np.all(np.isfinite(y)):
                slope, intercept, r, p, se = linregress(x, y)
                slopes[sr] = slope
    if slopes:
        S_avg = np.mean(list(slopes.values()))
        S_results[eps] = S_avg
        S_all_slopes[eps] = slopes
        Q_results[eps] = R_GAS * n_val * S_avg

print('=== Q（热变形激活能）计算结果 ===')
print(f'{"应变":>6s} | {"n":>8s} | {"S_avg":>12s} | {"Q (J/mol)":>14s} | {"Q (kJ/mol)":>12s}')
print('-' * 70)
for eps in strain_points:
    if eps in Q_results:
        print(f'{eps:>6.2f} | {n_results[eps]:>8.4f} | {S_results[eps]:>12.2f} | {Q_results[eps]:>14.2f} | {Q_results[eps]/1000:>12.2f}')

print()
print('各应变速率下的斜率 S 详情：')
for eps in [0.1, 0.3, 0.5]:
    if eps in S_all_slopes:
        print(f'  ε={eps}: ', ', '.join([f'ε̇={sr}:{s:.2f}' for sr, s in S_all_slopes[eps].items()]))

In [ ]:
# 可视化 Q 拟合 (ln[sinh(ασ)] vs 1/T)
rate_colors = plt.cm.viridis(np.linspace(0, 1, len(rates)))
fig, axes = plt.subplots(1, len(sample_strains), figsize=(5 * len(sample_strains), 4))

for ax, eps in zip(axes, sample_strains):
    alpha = alpha_results.get(eps)
    if alpha is None:
        continue
    for sr, c in zip(rates, rate_colors):
        sub = df_discrete[(df_discrete['真应变'] == eps) & (df_discrete['应变速率'] == sr)]
        if len(sub) >= 2:
            sigma = sub['流变应力'].values
            T_K = sub['温度_K'].values
            x = 1.0 / T_K * 1000  # 1000/T for readability
            y = np.log(np.sinh(alpha * sigma))
            if np.all(np.isfinite(y)):
                ax.scatter(x, y, color=c, label=f'ε̇={sr}', zorder=5)
                slope, intercept, _, _, _ = linregress(x, y)
                x_fit = np.linspace(x.min(), x.max(), 50)
                ax.plot(x_fit, slope * x_fit + intercept, '--', color=c, alpha=0.7)
    ax.set_xlabel('1000/T (K⁻¹)')
    ax.set_ylabel('ln[sinh(ασ)]')
    ax.set_title(f'Q 拟合 (ε={eps})')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 第四步：求解结构因子 ln A

$$\ln Z = \ln A + n \ln[\sinh(\alpha \sigma)]$$

其中 $Z = \dot{\varepsilon} \exp(Q/RT)$

以 ln[sinh(ασ)] 为横坐标，ln Z 为纵坐标进行全局线性回归，截距即为 ln A

In [ ]:
lnA_results = {}
lnA_regression_details = {}

for eps in strain_points:
    if eps not in alpha_results or eps not in n_results or eps not in Q_results:
        continue
    alpha = alpha_results[eps]
    Q_val = Q_results[eps]

    sub = df_discrete[df_discrete['真应变'] == eps].copy()
    if len(sub) < 3:
        continue

    sigma = sub['流变应力'].values
    T_K = sub['温度_K'].values
    sr_vals = sub['应变速率'].values

    ln_Z = np.log(sr_vals) + Q_val / (R_GAS * T_K)
    x = np.log(np.sinh(alpha * sigma))

    valid = np.isfinite(x) & np.isfinite(ln_Z)
    if valid.sum() >= 3:
        slope, intercept, r, p, se = linregress(x[valid], ln_Z[valid])
        lnA_results[eps] = intercept
        lnA_regression_details[eps] = {
            'slope': slope, 'intercept': intercept, 'r_squared': r**2,
            'n_from_regression': slope
        }

print('=== ln A 计算结果 ===')
print(f'{"应变":>6s} | {"ln A":>10s} | {"A":>14s} | {"R²":>8s} | {"回归斜率(≈n)":>12s}')
print('-' * 70)
for eps in strain_points:
    if eps in lnA_results:
        d = lnA_regression_details[eps]
        A_val = np.exp(lnA_results[eps]) if lnA_results[eps] < 700 else float('inf')
        print(f'{eps:>6.2f} | {lnA_results[eps]:>10.4f} | {A_val:>14.4e} | {d["r_squared"]:>8.4f} | {d["slope"]:>12.4f}')

In [ ]:
# 可视化 ln A 拟合 (ln Z vs ln[sinh(ασ)])
fig, axes = plt.subplots(1, len(sample_strains), figsize=(5 * len(sample_strains), 4))

for ax, eps in zip(axes, sample_strains):
    alpha = alpha_results.get(eps)
    Q_val = Q_results.get(eps)
    if alpha is None or Q_val is None:
        continue
    sub = df_discrete[df_discrete['真应变'] == eps]
    sigma = sub['流变应力'].values
    T_K = sub['温度_K'].values
    sr_vals = sub['应变速率'].values
    ln_Z = np.log(sr_vals) + Q_val / (R_GAS * T_K)
    x = np.log(np.sinh(alpha * sigma))
    valid = np.isfinite(x) & np.isfinite(ln_Z)

    # 不同温度不同颜色
    for temp, c in zip(temps, colors):
        mask = valid & (sub['温度_C'].values == temp)
        if mask.sum() > 0:
            ax.scatter(x[mask], ln_Z[mask], color=c, label=f'{temp}°C', zorder=5)

    slope, intercept, _, _, _ = linregress(x[valid], ln_Z[valid])
    x_fit = np.linspace(x[valid].min(), x[valid].max(), 50)
    ax.plot(x_fit, slope * x_fit + intercept, 'k--', label=f'线性拟合', alpha=0.8)
    ax.set_xlabel('ln[sinh(ασ)]')
    ax.set_ylabel('ln Z')
    ax.set_title(f'ln A 拟合 (ε={eps})\nlnA={intercept:.2f}, R²={linregress(x[valid], ln_Z[valid]).rvalue**2:.4f}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 汇总所有应变点的参数

In [ ]:
params_df = pd.DataFrame({
    '真应变': [eps for eps in strain_points if eps in alpha_results and eps in n_results and eps in Q_results and eps in lnA_results],
    'α': [alpha_results[eps] for eps in strain_points if eps in alpha_results and eps in n_results and eps in Q_results and eps in lnA_results],
    'n': [n_results[eps] for eps in strain_points if eps in alpha_results and eps in n_results and eps in Q_results and eps in lnA_results],
    'Q (J/mol)': [Q_results[eps] for eps in strain_points if eps in alpha_results and eps in n_results and eps in Q_results and eps in lnA_results],
    'Q (kJ/mol)': [Q_results[eps]/1000 for eps in strain_points if eps in alpha_results and eps in n_results and eps in Q_results and eps in lnA_results],
    'ln A': [lnA_results[eps] for eps in strain_points if eps in alpha_results and eps in n_results and eps in Q_results and eps in lnA_results],
})

print('=== 所有应变点的 Arrhenius 参数汇总 ===')
print(params_df.to_string(index=False))

## 9. 第三章：多项式拟合（参数与应变的连续化）

使用6次多项式拟合 α(ε), n(ε), Q(ε), ln A(ε)

In [ ]:
poly_degree = 6
eps_arr = params_df['真应变'].values

# 拟合各参数
poly_alpha = np.polyfit(eps_arr, params_df['α'].values, poly_degree)
poly_n = np.polyfit(eps_arr, params_df['n'].values, poly_degree)
poly_Q = np.polyfit(eps_arr, params_df['Q (J/mol)'].values, poly_degree)
poly_lnA = np.polyfit(eps_arr, params_df['ln A'].values, poly_degree)

print('=== 6次多项式拟合系数 ===')
coeff_labels = [f'x^{i}' for i in range(poly_degree, -1, -1)]

print('\nα(ε) 系数 (B₆ → B₀):')
for label, coeff in zip(coeff_labels, poly_alpha):
    print(f'  {label}: {coeff:.10e}')

print('\nn(ε) 系数 (C₆ → C₀):')
for label, coeff in zip(coeff_labels, poly_n):
    print(f'  {label}: {coeff:.10e}')

print('\nQ(ε) 系数 (D₆ → D₀):')
for label, coeff in zip(coeff_labels, poly_Q):
    print(f'  {label}: {coeff:.10e}')

print('\nln A(ε) 系数 (E₆ → E₀):')
for label, coeff in zip(coeff_labels, poly_lnA):
    print(f'  {label}: {coeff:.10e}')

In [ ]:
# 可视化多项式拟合效果
eps_fine = np.linspace(eps_arr.min(), eps_arr.max(), 200)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

param_names = ['α', 'n', 'Q (kJ/mol)', 'ln A']
param_data = [params_df['α'].values, params_df['n'].values,
              params_df['Q (kJ/mol)'].values, params_df['ln A'].values]
param_polys = [poly_alpha, poly_n, poly_Q, poly_lnA]
param_scales = [1, 1, 1/1000, 1]  # Q 需要从 J/mol 转为 kJ/mol 显示

for ax, name, data, poly, scale in zip(axes.flat, param_names, param_data, param_polys, param_scales):
    ax.scatter(eps_arr, data, c='red', s=50, zorder=5, label='计算值')
    fitted = np.polyval(poly, eps_fine) * scale
    ax.plot(eps_fine, fitted, 'b-', linewidth=2, label=f'{poly_degree}次多项式拟合')
    ax.set_xlabel('真应变 ε')
    ax.set_ylabel(name)
    ax.set_title(f'{name} vs ε')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. 第四章：全曲线预测

$$\sigma = \frac{1}{\alpha(\varepsilon)} \ln \left\{ \left(\frac{Z}{A(\varepsilon)}\right)^{\frac{1}{n(\varepsilon)}} + \sqrt{ \left(\frac{Z}{A(\varepsilon)}\right)^{\frac{2}{n(\varepsilon)}} + 1 } \right\}$$

其中 $Z = \dot{\varepsilon} \exp\left( \frac{Q(\varepsilon)}{RT} \right)$

In [ ]:
def predict_stress(epsilon, strain_rate, T_K, poly_alpha, poly_n, poly_Q, poly_lnA):
    """根据 Arrhenius 本构方程预测流变应力"""
    alpha = np.polyval(poly_alpha, epsilon)
    n_val = np.polyval(poly_n, epsilon)
    Q_val = np.polyval(poly_Q, epsilon)
    lnA = np.polyval(poly_lnA, epsilon)

    ln_Z = np.log(strain_rate) + Q_val / (R_GAS * T_K)
    # Z/A = exp(lnZ - lnA)
    ln_Z_over_A = ln_Z - lnA
    # (Z/A)^(1/n)
    x = np.exp(ln_Z_over_A / n_val)
    # σ = (1/α) * arcsinh(x) = (1/α) * ln(x + sqrt(x² + 1))
    sigma = (1.0 / alpha) * np.log(x + np.sqrt(x**2 + 1))
    return sigma

# 验证：在离散点处预测并与实验值对比
print('=== 模型预测 vs 实验值（离散应变点处） ===')
predictions = []
for _, row in df_discrete.iterrows():
    eps = row['真应变']
    if eps < eps_arr.min() or eps > eps_arr.max():
        continue
    sigma_pred = predict_stress(eps, row['应变速率'], row['温度_K'],
                                poly_alpha, poly_n, poly_Q, poly_lnA)
    if np.isfinite(sigma_pred) and sigma_pred > 0:
        predictions.append({
            '温度_C': row['温度_C'], '应变速率': row['应变速率'],
            '真应变': eps, '实验值': row['流变应力'],
            '预测值': sigma_pred, '误差%': abs(sigma_pred - row['流变应力']) / row['流变应力'] * 100
        })

pred_df = pd.DataFrame(predictions)
print(f'\n平均相对误差(AARE): {pred_df["误差%"].mean():.2f}%')
print(f'最大相对误差: {pred_df["误差%"].max():.2f}%')
print(f'相关系数 R: {np.corrcoef(pred_df["实验值"], pred_df["预测值"])[0,1]:.6f}')
print()
print('各条件下的预测对比（ε=0.3 处）：')
sample = pred_df[np.isclose(pred_df['真应变'], 0.3)]
print(sample[['温度_C', '应变速率', '实验值', '预测值', '误差%']].to_string(index=False))

In [ ]:
# 全曲线预测可视化
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
eps_pred = np.linspace(eps_arr.min(), eps_arr.max(), 100)

for ax, temp in zip(axes.flat, temps):
    for sr, c in zip(rates, rate_colors):
        # 实验数据
        exp = df_all[(df_all['温度_C'] == temp) & (df_all['应变速率'] == sr)].sort_values('真应变')
        exp = exp[(exp['真应变'] >= eps_arr.min()) & (exp['真应变'] <= eps_arr.max())]
        if len(exp) > 0:
            ax.plot(exp['真应变'], exp['流变应力'], '-', color=c, alpha=0.4, linewidth=1)

        # 预测曲线
        T_K = temp + 273.15
        sigma_pred = [predict_stress(e, sr, T_K, poly_alpha, poly_n, poly_Q, poly_lnA) for e in eps_pred]
        sigma_pred = np.array(sigma_pred)
        valid = np.isfinite(sigma_pred) & (sigma_pred > 0)
        if valid.sum() > 0:
            ax.plot(eps_pred[valid], sigma_pred[valid], '--', color=c, linewidth=2, label=f'ε̇={sr}')

    ax.set_title(f'T = {temp}°C')
    ax.set_xlabel('真应变')
    ax.set_ylabel('流变应力 (MPa)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Arrhenius 模型预测（虚线）vs 实验数据（实线）', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 预测值 vs 实验值散点图（评估模型精度）
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pred_df['实验值'], pred_df['预测值'], alpha=0.5, s=20, edgecolors='none')

max_val = max(pred_df['实验值'].max(), pred_df['预测值'].max()) * 1.05
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='y = x')

R_val = np.corrcoef(pred_df['实验值'], pred_df['预测值'])[0, 1]
AARE = pred_df['误差%'].mean()
ax.text(0.05, 0.9, f'R = {R_val:.4f}\nAARE = {AARE:.2f}%',
        transform=ax.transAxes, fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('实验值 (MPa)', fontsize=12)
ax.set_ylabel('预测值 (MPa)', fontsize=12)
ax.set_title('Arrhenius 模型精度评估', fontsize=14)
ax.legend()
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n=== 模型评估指标 ===')
print(f'相关系数 R: {R_val:.6f}')
print(f'决定系数 R²: {R_val**2:.6f}')
print(f'平均相对误差 AARE: {AARE:.2f}%')